In [39]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap
from sklearn.cluster import KMeans

df = pd.read_csv('log_dataset.csv')

cus = df[df['Type'] == 'Customer']
hub = df[df['Type'] == 'Hub']

coords = cus[['Latitude','Longitude']].values
weights = cus['Demand'].astype(int).values

points = np.repeat(coords, weights, axis=0)

model = KMeans(n_clusters=3, random_state=0)
model.fit(points)

centers = model.cluster_centers_

m = folium.Map(location=[10.78,106.70], zoom_start=11)

heat = cus[['Latitude','Longitude','Revenue']].values.tolist()
HeatMap(heat).add_to(m)

# khách hàng (icon đen nhỏ)
for _, r in cus.iterrows():
    folium.CircleMarker(
        [r['Latitude'], r['Longitude']],
        radius=3,
        color='black',
        fill=True,
        fill_opacity=0.3
    ).add_to(m)

# kho hiện tại (icon xanh lá khác)
for _, r in hub.iterrows():
    folium.Marker(
        [r['Latitude'], r['Longitude']],
        popup="Kho hien tai",
        icon=folium.Icon(color='green', icon='ok-sign')
    ).add_to(m)

# kho AI đề xuất (đám mây nền tím)
for c in centers:
    folium.Marker(
        [c[0], c[1]],
        popup="Kho de xuat",
        icon=folium.Icon(color='purple', icon='cloud')
    ).add_to(m)

m

**Nhận xét:**

1. Bài toán
Doanh nghiệp có nhiều khách hàng phân bố ở các khu vực khác nhau nhưng vị trí kho hiện tại có thể chưa tối ưu.
Mục tiêu là tìm vị trí đặt thêm kho sao cho gần khu vực có nhu cầu cao, giúp giảm quãng đường giao hàng.

2. Dữ liệu
Sử dụng file log_dataset.csv gồm:
- Tọa độ (Latitude, Longitude)
- Demand (nhu cầu)
- Revenue (doanh thu)

3. Phương pháp AI
Sử dụng K-Means để phân cụm vị trí khách hàng.
Để phản ánh nhu cầu, mỗi điểm được lặp lại theo Demand.
Nhờ đó các cụm sẽ tập trung vào khu vực có nhiều đơn hàng hơn.

4. Trực quan hóa
- Heatmap: thể hiện khu vực có doanh thu cao
- Marker xanh: kho hiện tại
- Marker đỏ: vị trí kho đề xuất
- Chấm xám: vị trí khách hàng

5. Giá trị ứng dụng
- Giảm chi phí vận chuyển
- Giao hàng nhanh hơn
- Hỗ trợ ra quyết định đặt kho dựa trên dữ liệu